# Live monitor

Read-only: training runs in tmux (`bash scripts/run_queue.sh ...`), never in this notebook,
so closing the browser or losing the SSH tunnel cannot kill a run. Re-run cells to refresh.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # repo root, for rlvr.*
import numpy as np
import matplotlib.pyplot as plt

ART = Path(os.environ.get('ARTIFACT_ROOT', Path.home() / 'artifacts'))
USD_PER_HOUR = 1.006   # g5.xlarge on-demand, us-east-1

def load_jsonl(p):
    return [json.loads(l) for l in Path(p).read_text().splitlines() if l.strip()] if Path(p).exists() else []

runs = {str(d.relative_to(ART / 'runs')): d for d in sorted((ART / 'runs').glob('*/*/seed*'))}
print(subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)
for name, d in runs.items():
    status = 'DONE' if (d / 'DONE.json').exists() else 'in progress'
    print(f'{name:40s} {status:12s} logged steps: {len(load_jsonl(d / "train_log.jsonl"))}')

## Progress, ETA and cost per run

In [ ]:
from rlvr.config import TIER_SETTINGS

for name, d in runs.items():
    recs = load_jsonl(d / 'train_log.jsonl')
    if len(recs) < 2:
        continue
    cfg = json.loads((d / 'config.json').read_text())
    total = cfg['steps_override'] or TIER_SETTINGS[cfg['compute_tier']]['training_steps']
    sec = float(np.median([r['step_seconds'] for r in recs[-20:]]))
    done = recs[-1]['step'] + 1
    left_h = sec * (total - done) / 3600
    print(f'{name:40s} step {done}/{total}  {sec:5.1f}s/step  ETA {left_h:5.1f} h  '
          f'run cost ~${sec * total / 3600 * USD_PER_HOUR:.2f}')

## Training curves
Mode entropy is logged for every condition (vanilla's collapse is the point of comparison); `frac_prompts_selected` only for SELF runs.

In [ ]:
keys = ['correctness_success_rate', 'mean_group_mode_entropy', 'frac_groups_multi_mode', 'approx_kl',
        'mean_pos_weight', 'frac_prompts_selected', 'frac_zero_var_groups', 'mean_completion_tokens',
        'frac_truncated', 'grad_norm', 'lr', 'step_seconds']
fig, axes = plt.subplots(3, 4, figsize=(18, 10))
for ax, key in zip(axes.flat, keys):
    for name, d in runs.items():
        recs = [r for r in load_jsonl(d / 'train_log.jsonl') if key in r]
        if recs:
            ax.plot([r['step'] for r in recs], [r[key] for r in recs], label=name, lw=1)
    ax.set_title(key, fontsize=10)
axes.flat[0].legend(fontsize=7)
plt.tight_layout()

## Where the time goes (latest logged step per run)

In [ ]:
for name, d in runs.items():
    recs = load_jsonl(d / 'train_log.jsonl')
    if recs:
        r = recs[-1]
        parts = {k[:-2]: r[k] for k in r if k.endswith('_s')}
        tot = sum(parts.values()) or 1
        print(f'{name:40s} ' + '  '.join(f'{k} {100 * v / tot:4.0f}%' for k, v in parts.items()))

## In-training monitor evals (50 MATH-500 problems x 10 samples vs base model)

In [ ]:
for name, d in runs.items():
    for e in load_jsonl(d / 'eval_log.jsonl'):
        print(f"{name:40s} step {e['step']:4d}  slope {e['shrinkage_slope']:+.4f}  Pass@k {e['pass_at_k']}")

## Recent log lines

In [ ]:
for name, d in runs.items():
    log = d / 'stdout.log'
    if log.exists():
        print(f'--- {name}')
        print(''.join(log.read_text().splitlines(keepends=True)[-5:]))